<a href="https://colab.research.google.com/github/Jackson24x/Datasets/blob/analysis_of_housing_data_using_logistic_regression/analysis_of_housing_data_using_logistic_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Introduction and Objective of the Analysis

The objective of this analysis is to explore a dataset containing information about houses in the USA, perform exploratory data analysis (EDA), build a logistic regression model to classify houses as 'High Value' based on their price, and improve the model through hyperparameter tuning and feature standardization. The goal is to gain insights into the factors that contribute to a house being considered 'High Value' and to predict this classification for new data

### 2. Data Loading and Preliminary Exploration

loading the dataset and performing some basic exploration to understand its structure

In [ ]:
import pandas as pd

# Load the dataset
housing_data = pd.read_csv(r'C:\Users\kicch\Downloads\DATA_vista\USA_Housing.csv')

# Display the first five rows of the dataset to understand its structure
housing_data.head()

The dataset contains information about houses, including average area income, house age, number of rooms and bedrooms, area population, price, and address.

### 3. Exploratory Data Analysis (EDA)

#### Statistical Summary

Let's start the EDA by generating a statistical summary of the numerical features in the

In [ ]:
housing_data.describe()

The statistical summary provides insights into the distribution of each numerical feature, including the mean, standard deviation, and quartile values.

#### Visualization of Distributions

Next, let's visualize the distributions of these features to better understand their spread and skew

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Set the aesthetic style of the plots
color_palette = "Set2"
sns.set(style="whitegrid", palette=color_palette)

# Plotting distributions of numerical features
fig, axes = plt.subplots(3, 2, figsize=(16, 12))

sns.histplot(housing_data['Avg. Area Income'], kde=True, ax=axes[0, 0])
axes[0, 0].set_title('Distribution of Avg. Area Income')

sns.histplot(housing_data['Avg. Area House Age'], kde=True, ax=axes[0, 1])
axes[0, 1].set_title('Distribution of Avg. Area House Age')

sns.histplot(housing_data['Avg. Area Number of Rooms'], kde=True, ax=axes[1, 0])
axes[1, 0].set_title('Distribution of Avg. Area Number of Rooms')

sns.histplot(housing_data['Avg. Area Number of Bedrooms'], kde=True, ax=axes[1, 1])
axes[1, 1].set_title('Distribution of Avg. Area Number of Bedrooms')

sns.histplot(housing_data['Area Population'], kde=True, ax=axes[2, 0])
axes[2, 0].set_title('Distribution of Area Population')

sns.histplot(housing_data['Price'], kde=True, ax=axes[2, 1])
axes[2, 1].set_title('Distribution of Price')

plt.tight_layout()
plt.show()

The distributions of the features have been visualized, showing a variety of shapes and spreads. This helps us understand the data's characteristics better, such as the skewness in the distribution of prices and area income.

#### Outlier Detection and Removal

Next, we will identify outliers in key columns using the Interquartile Range (IQR) method. Let's focus on "Avg. Area Income" and "Price" as examples

In [ ]:
def detect_outliers(data, feature):
    Q1 = data[feature].quantile(0.25)
    Q3 = data[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = data[(data[feature] < lower_bound) | (data[feature] > upper_bound)]
    return outliers

# Detecting outliers for 'Avg. Area Income'
outliers_avg_income = detect_outliers(housing_data, 'Avg. Area Income')

# Detecting outliers for 'Price'
outliers_price = detect_outliers(housing_data, 'Price')

(len(outliers_avg_income), len(outliers_price))

There are 32 outliers detected in the "Avg. Area Income" feature and 35 outliers in the "Price" feature.

#### Feature Engineering

Creating a new feature that might reveal interesting insights. Let's create a "House Value per Room" feature by dividing the "Price" by the "Avg. Area Number of Rooms

In [ ]:
# Creating a new feature 'House Value per Room'
housing_data['House Value per Room'] = housing_data['Price'] / housing_data['Avg. Area Number of Rooms']

# Display the first few rows to confirm the new feature has been added
housing_data.head()

new feature "House Value per Room" has been created.

#### Multivariate Analysis

To further explore relationships between variables,we will create a pair plot for a subset of the data. Given the size of the dataset, we'll sample 500 data points for visualization(Sampling of data is done)

In [ ]:
# Sample 500 data points for visualization
df_sample = housing_data.sample(500, random_state=42)

# Create pair plot
sns.pairplot(df_sample, vars=['Avg. Area Income', 'Avg. Area House Age', 'Avg. Area Number of Rooms', 'Area Population', 'Price', 'House Value per Room'])
plt.show()

### 4. Pre-processing for Model Building
#### Creating a Binary Target Variable
classification of houses as 'High Value' based on their price, considering a house High Value if its price is above the median price

In [ ]:
from sklearn.model_selection import train_test_split

# Creating a binary target variable 'High Value'
housing_data['High Value'] = housing_data['Price'] > housing_data['Price'].median()

# Splitting the dataset into features and target variable
X = housing_data.drop(['High Value', 'Address'], axis=1)
y = housing_data['High Value']

# Splitting the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### Feature Standardization
Standardizing the features to have a mean of 0 and a standard deviation of 1, which is crucial for models like logistic regression

In [ ]:
from sklearn.preprocessing import StandardScaler

# Standardizing the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

The features have been standardized for both the training and test sets.

### 5. Model Building

#### Logistic Regression Model Creation

building a logistic regression model to classify houses as 'High

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Initializing and fitting the Logistic Regression model
logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_train_scaled, y_train)

# Making predictions
y_pred = logistic_model.predict(X_test_scaled)

# Evaluating the model
report = classification_report(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

(report, conf_matrix)

The logistic regression model achieved an accuracy of 99% with precision, recall, and F1-score all around 99% for both classes (High Value and not High Value). The confusion matrix shows that the model correctly classified 474 as not High Value and 518 as High Value, with only 5 false positives and 3 false negatives.

### 6. Model Improvement

#### Hyperparameter Tuning and Cross-Validation

will perform hyperparameter tuning using GridSearchCV with cross-validation to find the optimal parameters for the logistic regression

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# Define the parameter grid
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']
}

# Create a GridSearchCV object
grid_search = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5, scoring='accuracy')

# Fit the model to the training data
grid_search.fit(X_train_scaled, y_train)

# Find the best parameters and the best accuracy
best_params = grid_search.best_params_
best_accuracy = grid_search.best_score_

# Predict on the test set with the best parameters
y_pred_best = grid_search.predict(X_test_scaled)
best_test_accuracy = accuracy_score(y_test, y_pred_best)

(best_params, best_accuracy, best_test_accuracy)

checking accuracy score

In [ ]:
from sklearn.metrics import accuracy_score

# Predict on the test set with the best parameters
y_pred_best = grid_search.predict(X_test_scaled)
best_test_accuracy = accuracy_score(y_test, y_pred_best)

(best_params, best_accuracy, best_test_accuracy)

After hyperparameter tuning and cross-validation, the best regularization strength (`C`) was found to be 100, and the best solver was `newton-cg`. The accuracy on the training set (averaged across the cross-validation splits) improved to approximately 99.8%, and the accuracy on the test set improved to 99.7%.

#### Model Interpretation

Let's interpret the model's coefficients to understand the importance and impact of different features on the

In [ ]:
# Extracting the coefficients from the best model
coefficients = grid_search.best_estimator_.coef_[0]

# Creating a DataFrame to display feature names and their corresponding coefficients
feature_names = X.columns
coef_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefficients})

# Sorting the DataFrame by the absolute values of coefficients in descending order
display_coef_df = coef_df.sort_values(by='Coefficient', key=abs, ascending=False)

# Display the sorted DataFrame
display_coef_df

The model's coefficients reveal the following insights about the importance and impact of different features on predicting whether a house is considered 'High Value':

- **Price**: Has the highest positive coefficient, indicating a strong positive correlation with the likelihood of a house being classified as 'High Value'. This is expected as the target variable was derived based on the house price.
- **House Value per Room**: Also has a significant positive coefficient, suggesting that as the value per room increases, so does the likelihood of a house being classified as 'High Value'.
- **Avg. Area Number of Rooms**: Has a positive coefficient, indicating a smaller but still positive impact on the classification as 'High Value'.
- **Avg. Area House Age** and **Avg. Area Income**: Have smaller positive coefficients, suggesting they have a lesser but still positive influence on the likelihood of a house being classified as 'High Value'.
- **Avg. Area Number of Bedrooms** and **Area Population**: Have negative coefficients, indicating that as these values increase, the likelihood of a house being classified as 'High Value' slightly decreases, although their impact is relatively minor compared to other features.

### 7. Conclusion and Future Steps

This analysis explored a dataset of USA houses, performed EDA, built a logistic regression model to classify houses as 'High Value', and improved the model through hyperparameter tuning and feature standardization. The model showed high accuracy in classification, and model interpretation provided insights into the factors influencing a house's classification as 'High Value'.

Future steps could include exploring other machine learning models, further feature engineering, or deploying the model for real-world applications.